# 18 — Evaluate motion information from the saved GAVD grid

This notebook loads the **same five-fold, five-seed grid trained in
Notebook 17**, including real train/test source membership. It never
starts encoder training. In a fresh kernel, it reconstructs expected
data/model identities, locates every saved job, and reuses or recomputes
its frozen readouts. A missing job produces an explicit status table;
a partial grid cannot become a full-grid result.

The central question from [TUTORIAL.md](docs/TUTORIAL.md) is whether
training learns useful movement information that simple averaging hides.
We compare temporal summaries against the same summaries of initial
features, then inspect predictor correspondence separately. This makes
favorable and unfavorable results equally interpretable.

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib_inline.backend_inline import set_matplotlib_formats

def locate_suite():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "neurips-laterality"):
            if (candidate / "laterality_extensions/motion_structured_masks.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Run from the research project directory.")

SUITE_ROOT = locate_suite()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))
from laterality_extensions.motion_structured_masks import (
    StudyArm, mamp_logits, sample_study_mask, study_arms,
    paired_study_masks, context_cue_audit,
)
from laterality_extensions.comparative_masks import motion_scores
from notebook_progress import (
    NotebookTaskProgress, run_notebook_task, study_inputs_with_progress,
    audit_training_masks_with_progress, grid_status_with_progress,
    run_gavd_grid_with_progress, collect_gavd_grid_with_progress,
    evaluate_retained_motion_with_progress,
)
set_matplotlib_formats("svg", "png")
pd.set_option("display.precision", 3)

## 1. Load the real GAVD cohort and declare the full split/seed grid

Run cells in order in a Python kernel with the project's dependencies.
Long tasks use the shared `notebook_progress.py` wrapper: one updating
display shows the current stage, fold/seed, elapsed time and estimated
remaining time. Mask batches and optimizer updates appear within their
active stage. ETA adjusts as stages finish; their costs differ. Cached,
disabled, missing-input and failed tasks receive explicit status labels.
`DATA_MODE="gavd"` is the default. The helper below follows the same
preparation and source splitting functions as notebooks 01 and 02:

1. Verify an existing paper-profile cohort and split manifest by their
   content hashes. If absent, read the local GAVD pose archives and
   official annotations, apply the existing QC and target rules, and
   create those two artifacts. The first run takes longer.
2. The protocol fixes 642 pose archives and 666 annotations. If the local
   cache contains later additions, recover the original extraction
   generations only when their inventory **exactly** matches the locked
   count and SHA-256. Store verified copies under the paper artifact
   root. The original cache stays intact. A mismatch stops with an
   actionable error; it cannot silently switch to generated data.
3. The reference QC result is 625 clips from 93 source videos. Inputs have
   shape `[clips, 64, 33, 3]`; four prepared steps form each of 16 tokens
   per landmark. Naturally missing observations remain marked invalid.
   The target is a coordinate-derived bilateral movement contrast, not
   the dataset's condition annotation.
4. Reuse the five video-disjoint outer folds. Seeds 42--46 change
   initialization, source draws, augmentations and masks; they repeat
   the **same** train/test partitions. A video is held out exactly once
   per seed. There are 25 fold/seed combinations, not 25 independent
   datasets. Video separation does not establish subject separation.

The printed census makes train and test membership visible. Source IDs
and sequence IDs are retained in `inputs["memberships"]`. The reference
train/test clip counts are 436/189, 443/182, 553/72, 548/77 and 520/105.
Unequal clip counts are expected because entire videos stay together.

A small generated-data path remains available only through the explicit
`LATERALITY_MOTION_DATA_MODE=synthetic` software-check setting. It prints
its reduced scope and uses a separate artifact directory. It provides
no GAVD results. See [the run guide](docs/MOTION_GAVD_WORKFLOW.md) for
paths, environment settings, recovery and a notebook-by-notebook walkthrough.

In [ ]:
from laterality_extensions.motion_gavd import gavd_plan, readout_contrasts
DATA_MODE = os.getenv("LATERALITY_MOTION_DATA_MODE", "gavd")
FOLDS = (0, 1, 2, 3, 4)
SEEDS = (42, 43, 44, 45, 46)
EXPERIMENTS = tuple(os.getenv("LATERALITY_MOTION_EXPERIMENTS", "motion,regions").split(","))
CREATE_MISSING_INPUTS = True
DEVICE = os.getenv("LATERALITY_DEVICE", "auto")
# Same explicit training switch as Notebook 12; also accept the study-specific alias.
RUN_TRAINING = os.getenv("LATERALITY_MOTION_RUN_REAL",
                        os.getenv("LATERALITY_RESEARCH_RUN_REAL", "0")) == "1"
if DATA_MODE == "synthetic":
    FOLDS, SEEDS = (0,), (42,)
    print("EXPLICIT SYNTHETIC SOFTWARE CHECK: one fold/seed, one update, no GAVD evidence")
OUTPUT_ROOT = Path(os.getenv("LATERALITY_MOTION_OUTPUT_ROOT", str(SUITE_ROOT / "artifacts" /
    ("motion_structured" if DATA_MODE == "gavd" else "motion_structured_synthetic"))))
print(f"Mode={DATA_MODE}; folds={FOLDS}; seeds={SEEDS}; device={DEVICE}")
print(f"Training enabled={RUN_TRAINING}; outputs={OUTPUT_ROOT}")

In [ ]:
input_progress = NotebookTaskProgress("Dataset preparation and source splits", "stage")
inputs = study_inputs_with_progress(mode=DATA_MODE, folds=FOLDS, seeds=SEEDS,
    create_missing=CREATE_MISSING_INPUTS, progress=input_progress)
display(inputs["census"])
assert inputs["census"].source_overlap.eq(0).all()
display(inputs["memberships"].head(8))
if DATA_MODE == "gavd":
    cohort = inputs["cohort"]
    display(cohort.table.groupby("condition").agg(
        accepted_clips=("sequence_id", "size"), source_videos=("video_id", "nunique")))
    display(pd.Series({key: cohort.attrition[key] for key in
        ("input_sequences", "accepted_sequences", "accepted_sources", "excluded_sequences")}))
    print("Cohort:", cohort.cohort_digest)
    print("Split:", inputs["splits"]["split_digest"])
    print("Artifacts:", inputs["context"].artifact_root)

## 2. Reconstruct the exact training grid before loading results

Keep `DATA_MODE`, `FOLDS`, `SEEDS`, `EXPERIMENTS`, `DEVICE` and
`OUTPUT_ROOT` identical to Notebook 17. The `RUN_TRAINING` variable has
no effect in this notebook. Model initialization settings and the resolved
backend are part of compatibility; changing them identifies a different
job. A saved manifest is not its own proof of compatibility.

The primary declaration expects 50 paired jobs containing 125 encoders.
The census verifies test coverage independently of the prediction files.
With all five folds, every accepted sequence must appear in a test set
exactly once per seed. Changing the scope to a subset creates a pilot.

In [ ]:
plan = gavd_plan(inputs, experiments=EXPERIMENTS, device=DEVICE, output_dir=OUTPUT_ROOT)
display(pd.Series({key: plan[key] for key in
    ("scope", "training_runs", "optimizer_updates", "recipe_source", "output_dir")}))
display(pd.Series(plan["settings"], name="Declared training settings"))
display(plan["workload"].groupby(["experiment", "fold", "seed"], sort=False).agg(
    encoders=("condition", "size"), optimizer_updates=("updates", "sum")))
display(pd.DataFrame([{"experiment": e, **arm} for e, arms in plan["arms"].items()
                      for arm in arms.values()]))

In [ ]:
checkpoint_progress = NotebookTaskProgress("Saved GAVD checkpoint inspection", "stage")
status = grid_status_with_progress(plan, inputs, progress=checkpoint_progress)
display(status)
availability = status.groupby(["experiment", "training_status"]).size().unstack(fill_value=0)
ax = availability.plot.bar(stacked=True, figsize=(8, 3.5), title=f"{DATA_MODE.upper()}: saved paired jobs")
ax.set(xlabel="Experiment", ylabel="Fold/seed jobs")
ax.figure.tight_layout(); display(ax.figure); plt.close(ax.figure)

## 3. Separate representation learning from readout design

| Representation | What is frozen? | Inference it supports |
|---|---|---|
| `pretrained_online__mean` / `__mean_motion` | Trained online encoder | Content available without the JEPA predictor |
| `pretrained_teacher__mean` / `__mean_motion` | Final EMA teacher | Teacher representation quality |
| `initial_online__mean` / `__mean_motion` | Same job's initial encoder | Architecture and readout control |
| `direct_pose` | Prepared coordinates | Information already accessible from pose summaries |
| `training_mean` | Training-source target mean | No-feature prediction baseline |

`mean` retains the existing five bilateral sums and differences of
average features. `mean_motion` also includes temporal standard
deviation, mean absolute consecutive feature changes and valid-support
fractions. Only common bilateral support and adjacent valid transitions
contribute. The summaries describe prepared tokens; they cannot restore
physical timing lost during input resizing.

The ridge grid is 0.01, 0.1, 1, 10, 100, 1,000 and 10,000. Preprocessing
and alpha selection use outer-training sources only. This extension uses
the comparative readout helper's **three source-separated inner groups**;
these are not the registered protocol's four inner model-selection
folds. Pretraining sees all outer-training sources, so this inner step
selects the readout only. Tuning an entire encoder recipe would require
excluding inner-validation sources from candidate pretraining too.

Both summaries are declared outputs. Outer-test scores cannot choose a
summary, checkpoint or alpha. A boundary alpha is reported for inspection;
a wide grid alone does not guarantee adequate regularization.

In [ ]:
evaluation_progress = NotebookTaskProgress("Saved-grid readouts and source-level reporting", "stage")
results = collect_gavd_grid_with_progress(plan, inputs, progress=evaluation_progress)
print(results["status"])
COMPLETE = results["status"] == "Complete"
if COMPLETE:
    print("Verified grid report:", results["directory"])
    display(results["jobs"])
    display(results["per_seed"][["experiment", "condition", "representation", "seed",
        "r2", "mae", "evaluated_clips", "evaluated_sources"]])
else:
    display(results["jobs"].query("training_status == 'missing'"))
    print("Complete these jobs in Notebook 17, then rerun this cell. No substitute model was trained.")

## 4. Pool predictions correctly and inspect paired differences

R² and MAE are calculated after pooling all declared outer-test folds
for each seed, with equal total weight per source video. With the full
GAVD grid, each arm/summary/seed has 625 predictions from 93 videos.
Fold-specific R² has a different denominator and must not be averaged.
A negative pooled R² means worse squared error than the pooled weighted
test-target mean; the deployable training-mean control is shown separately.

For the tables below, positive `delta_r2` and negative `delta_mae` favor
the first representation in the contrast. Compare trained against initial
within the **same** summary before attributing gains to learning.
The saved `paired_intervals` additionally compares each mask arm with its
own uniform control using teacher mean-motion features and 2,000 paired
source-video bootstrap resamples. Whole videos, clips and paired seed
predictions stay together. Those intervals condition on fitted models;
they do not measure full retraining uncertainty. Seed variation is separate.

In [ ]:
if COMPLETE:
    display(results["summary"])
    contrasts = readout_contrasts(results["per_seed"])
    display(contrasts)
    display(contrasts.groupby(["experiment", "condition", "contrast"], sort=False).agg(
        mean_delta_r2=("delta_r2", "mean"), seed_sd_delta_r2=("delta_r2", "std"),
        mean_delta_mae=("delta_mae", "mean")))
    display(results["paired_intervals"])
    selected = results["selection"].query("selected")
    display(selected[["experiment", "condition", "representation", "fold", "seed",
                      "selected_alpha", "at_grid_boundary"]])
    display(results["diagnostics"][["experiment", "condition", "representation", "fold", "seed",
                                    "effective_rank", "near_constant"]])
else:
    print("No complete declared grid: performance and learned-over-initial inference remain pending.")

## 5. Ask whether the predictor preserves clip correspondence

Frozen encoder readout and JEPA prediction answer different questions.
Every saved job also uses the same prespecified evaluation masks (bank
seed 1801), including scattered targets and bilateral leg gaps. The normal
pathway is online encoder → predictor, with full-input teacher targets.
Initial-model controls undergo the same diagnostics.

Compare `mismatched_target_mse` against
`matched_target_mse_on_control_clips`: these use the same subset where a
cross-source mismatch exists. A larger mismatch error supports sensitivity
to clip correspondence. A smaller raw own-teacher MSE across two trained
arms cannot rank semantics, because their teacher spaces and scales differ.
`normalized_error` divides by mean squared teacher-channel value, not
centered variance. Inspect feature variation and effective rank as well.

In [ ]:
if COMPLETE:
    predictor = results["predictor_diagnostics"].copy()
    predictor["correspondence_gap"] = (
        predictor.mismatched_target_mse - predictor.matched_target_mse_on_control_clips)
    display(predictor[["experiment", "condition", "fold", "seed", "evaluation_mask",
        "evaluated_clips", "evaluated_sources", "feature_mse", "normalized_error",
        "matched_target_mse_on_control_clips", "mismatched_target_mse", "correspondence_gap"]])
else:
    print("Predictor diagnostics await the same saved GAVD grid; no synthetic scores substituted.")

## 6. Turn the evidence into the next experiment

| Observed pattern | Supported interpretation | Next investigation |
|---|---|---|
| Temporal summaries help initial and trained features similarly | Readout design explains much of the gain | Keep the matched initial control; avoid crediting JEPA alone |
| Trained-over-initial gain appears under temporal summaries | Averaging may hide learned movement content | Check seeds, source contrasts and regularization diagnostics |
| A mask beats its uniform control but not initial features | Relative masking improvement without demonstrated learning benefit | Inspect the objective and preparation before broader mixtures |
| Low predictor error accompanies near-constant features | Error reduction may reflect an uninformative target space | Inspect teacher variation and mismatch controls |
| Neither readout nor predictor gives useful contrasts | Current representation/task may miss the endpoint | Revisit timing preparation or a focused objective change |

These are conditional interpretations, not conclusions manufactured from
absent results. Report unfavorable outcomes, boundary penalties and
unstable seeds. GAVD has informed these hypotheses; even a full new grid
remains development evidence for a coordinate-derived endpoint.

Missing raw measurements require the before-preparation tests in Notebook
13. Future prediction requires Notebook 14's past-only inputs and
persistence, velocity, mismatched-future and observed-future controls.
Completion masks do not establish forecasting or clinical validity.

## 7. Optional teaching check: a signal that temporal averaging destroys

This last cell is explicitly generated data, separate from the GAVD grid.
Two sinusoidal limbs have different amplitudes but zero temporal means.
A temporal summary should recover their known amplitude contrast while a
mean cannot. This validates a recoverable construction; it does not prove
that real JEPA tokens encode amplitude in the same way.

In [ ]:
from laterality_extensions.motion_readout import pooling_positive_control
teaching_progress = NotebookTaskProgress("Generated amplitude control", "stage")
teaching_scores, generated = run_notebook_task(pooling_positive_control,
    progress=teaching_progress, label="Fit and evaluate the two generated-data summaries")
display(teaching_scores.assign(evidence="generated amplitude control, not GAVD"))
fig, ax = plt.subplots(figsize=(7, 2.8), constrained_layout=True)
ax.plot(generated["wave"], label="Unit oscillation (mean zero)")
ax.axhline(0, color="gray", linewidth=0.7)
ax.set(title="Generated teaching control only", xlabel="Prepared step", ylabel="Amplitude")
ax.legend(); display(fig); plt.close(fig)

## 8. Optional: reuse a retained Notebook 12 encoder first

If compatible older checkpoints are available, this is the cheaper
readout test recommended in the tutorial. Set
`LATERALITY_RETAINED_COMPARISON` to one Notebook 12 job directory.
The loader checks current data, code and runtime, then saves separate
readouts without changing the original checkpoint. This optional
single-job diagnostic cannot substitute for Notebook 17's full grid.

In [ ]:
retained_directory = os.getenv("LATERALITY_RETAINED_COMPARISON", "")
retained_progress = NotebookTaskProgress("Optional Notebook 12 readout reanalysis", "stage")
retained = evaluate_retained_motion_with_progress(retained_directory,
    progress=retained_progress, enabled=bool(retained_directory))
if retained_directory:
    display(retained["selection"].query("selected")[[
        "condition", "representation", "fold", "seed", "selected_alpha", "at_grid_boundary"]])
    print("One retained Notebook 12 job reanalysed without encoder training; separate from the full grid.")
else:
    print("Optional older-checkpoint reanalysis not configured.")